In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [6]:
DATA_ROOT = Path("movie_book_cdsr_processed/multimodal_features_last")
TRAIN_CSV = "movie_book_cdsr_processed/train.csv"
SAVE_PATH = Path("movie_book_cdsr_processed/user_preferences2/") 
SAVE_PATH.mkdir(exist_ok=True)

# 三个全局嵌入
ID_FEAT = DATA_ROOT / "item_id_collab_512.npy"
IMG_FEAT = DATA_ROOT / "image_features_512.npy"
TEXT_FEAT = DATA_ROOT / "text_features_clip_512.npy"

# 维度
D_MODEL = 512
NUM_HEADS = 8
MAX_LEN = 4096

### 加载全局嵌入向量、用户交互序列

In [7]:
E_id = torch.from_numpy(np.load(ID_FEAT)).float()
E_img = torch.from_numpy(np.load(IMG_FEAT)).float()
E_tex = torch.from_numpy(np.load(TEXT_FEAT)).float()

print("全局嵌入加载完成：")
print(f"E_id: {E_id.shape}")
print(f"E_img: {E_img.shape}")
print(f"E_tex: {E_tex.shape}")

"""
    从交互数据中生成每个用户的三类序列：
    - S^X：电影域交互序列（仅电影）
    - S^Y：图书域交互序列（仅图书）
    - S^{X+Y}：跨域融合序列（所有交互按时序排列）
    movie：0 book：1    
"""
def build_all_user_sequences():
    df = pd.read_csv(TRAIN_CSV)
    user_seqs = {}
    for user, g in df.groupby("user_id"):
        g = g.sort_values("timestamp")
        Sx = g[g["domain"] == 0]["item_id"].tolist()
        Sy = g[g["domain"] == 1]["item_id"].tolist()
        Sxy = g["item_id"].tolist()
        user_seqs[user] = (Sx, Sy, Sxy)
    return user_seqs

user_sequences = build_all_user_sequences()
all_user_ids = list(user_sequences.keys())
print(f"总用户数：{len(all_user_ids)}")

全局嵌入加载完成：
E_id: torch.Size([132015, 512])
E_img: torch.Size([132015, 512])
E_tex: torch.Size([132015, 512])
总用户数：20738


In [8]:
print(user_sequences[1])  # 打印第一个用户的三类序列示例

([18, 19, 20, 21], [13, 14, 15, 16, 17, 23, 22, 24, 25, 26, 27, 28, 29, 37, 35, 34, 36, 32, 31, 30, 33, 38, 39, 40, 41, 42, 43, 49, 48, 47, 45, 44, 46, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60], [13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 22, 24, 25, 26, 27, 28, 29, 37, 35, 34, 36, 32, 31, 30, 33, 38, 39, 40, 41, 42, 43, 49, 48, 47, 45, 44, 46, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60])


### 自注意力 + 多头自注意力编码

In [9]:
# 位置编码
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))
        pe[:,0::2] = torch.sin(pos * div)
        pe[:,1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)
    def forward(self, x):
        return x + self.pe[:x.size(1)]
    
# 因果多头自注意力（QKV + 缩放点积）
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.out = nn.Linear(d_model, d_model)
    def forward(self, x):
        B, T, C = x.shape
        # 多头形状 （B, T,3, n_head, head_dim）
        qkv = self.qkv(x).reshape(B, T, 3, self.n_head, self.head_dim).permute(2,0,3,1,4)
        q,k,v = qkv.unbind(0) # 每个形状 (B, T, 3*d_model)
        # 缩放点积注意力
        attn = (q @ k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        attn = attn.masked_fill(~mask, -1e9)
        attn = F.softmax(attn, dim=-1)
        # 拼接多头输出
        out = (attn @ v).transpose(1,2).reshape(B,T,C)
        return self.out(out)


# Transformer 编码器层
class TransformerLayer(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalMultiHeadAttention(d_model, n_head)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model,4*d_model),nn.GELU(),nn.Linear(4*d_model,d_model))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x
    
# 初始化 9 个独立编码器
# 0:X^ID, 1:X^视觉, 2:X^文本
# 3:Y^ID, 4:Y^视觉, 5:Y^文本
# 6:X+Y^ID, 7:X+Y^视觉, 8:X+Y^文本
encoders = nn.ModuleList([
    nn.Sequential(PositionalEncoding(D_MODEL), TransformerLayer(D_MODEL, NUM_HEADS)) 
    for _ in range(9)
])
encoders.eval() # 推理模式

ModuleList(
  (0-8): 9 x Sequential(
    (0): PositionalEncoding()
    (1): TransformerLayer(
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): CausalMultiHeadAttention(
        (qkv): Linear(in_features=512, out_features=1536, bias=True)
        (out): Linear(in_features=512, out_features=512, bias=True)
      )
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=2048, out_features=512, bias=True)
      )
    )
  )
)

### 用户编码函数

In [10]:
def sequence_to_emb(seq, E_id, E_img, E_tex):
    if not seq: return [None]*3
    seq = torch.tensor(seq, dtype=torch.long)
    return [E_id[seq], E_img[seq], E_tex[seq]]

def encode_one_user(seqs):
    Sx, Sy, Sxy = seqs
    embs = sequence_to_emb(Sx,E_id,E_img,E_tex) + sequence_to_emb(Sy,E_id,E_img,E_tex) + sequence_to_emb(Sxy,E_id,E_img,E_tex)
    prefs = []
    for i, emb in enumerate(embs):
        if emb is None:
            prefs.append(torch.zeros(1, D_MODEL))
            continue
        emb = emb.unsqueeze(0)
        feat = encoders[i](emb)[:, -1, :]
        prefs.append(feat)
    return torch.cat(prefs, dim=0).detach().numpy()   # 拼接9组偏好，转numpy保存

# 批量编码所有用户
print("批量生成所有用户偏好...")
all_user_prefs = []
for user in all_user_ids:
    pref = encode_one_user(user_sequences[user])
    all_user_prefs.append(pref)

# 保存
np.save(SAVE_PATH / "user_ids.npy", np.array(all_user_ids))          # 用户ID列表
np.save(SAVE_PATH / "user_9_preferences.npy", np.array(all_user_prefs)) # 所有用户9组偏好
print(f"保存完成！")
print(f"- 用户ID：{len(all_user_ids)}")
print(f"- 偏好矩阵形状：[用户数, 9, 512] = {np.array(all_user_prefs).shape}")

批量生成所有用户偏好...


保存完成！
- 用户ID：20738
- 偏好矩阵形状：[用户数, 9, 512] = (20738, 9, 512)
